# Data Cleaning & Standardization

## Objective

This notebook performs data cleaning and standardization on the Moroccanized datasets generated during the previous ETL stage.

The goal is to improve data consistency and ensure data quality.

In [41]:
from pathlib import Path
import pandas as pd
import numpy as np

In [42]:
PROJECT_ROOT = Path("..")

PROCESSED = PROJECT_ROOT / "data" / "processed"

In [43]:
def load_csv(filename):
    df = pd.read_csv(PROCESSED / filename)
    return df

In [44]:
customers = load_csv("customers.csv")
orders = load_csv("orders.csv")
order_items = load_csv("order_items.csv")
payments = load_csv("payments.csv")
products = load_csv("products.csv")
reviews = load_csv("reviews.csv")
sellers = load_csv("sellers.csv")
geolocation = load_csv("geolocation.csv")

# Standardizing Column Names

In [45]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("__", "_", regex=False)
    )
    return df

In [46]:
customers = clean_column_names(customers)
orders = clean_column_names(orders)
order_items = clean_column_names(order_items)
payments = clean_column_names(payments)
products = clean_column_names(products)
reviews = clean_column_names(reviews)
sellers = clean_column_names(sellers)
geolocation = clean_column_names(geolocation)

In [47]:
customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

# Standardize Text Values

In [48]:
def clean_text(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.strip()
              .str.replace(r"\s+", " ", regex=True)
    )

In [49]:
customers["customer_city"] = (
    clean_text(customers["customer_city"])
    .str.title()
)

customers["customer_state"] = (
    clean_text(customers["customer_state"])
    .str.upper()
)

In [50]:
sellers["seller_city"] = (
    clean_text(sellers["seller_city"])
    .str.title()
)

sellers["seller_state"] = (
    clean_text(sellers["seller_state"])
    .str.upper()
)

In [51]:
geolocation["geolocation_city"] = (
    clean_text(geolocation["geolocation_city"])
    .str.title()
)

geolocation["geolocation_state"] = (
    clean_text(geolocation["geolocation_state"])
    .str.upper()
)

In [52]:
products["product_category_name"] = (
    clean_text(products["product_category_name"])
    .str.title()
)

In [53]:
payments["payment_type"] = (
    clean_text(payments["payment_type"])
    .str.title()
)

In [54]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,25795,Casablanca,CASABLANCA-SETTAT
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,10860,Casablanca,CASABLANCA-SETTAT
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,86820,Casablanca,CASABLANCA-SETTAT
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,64886,Casablanca,CASABLANCA-SETTAT
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,16265,Meknès,CASABLANCA-SETTAT


# Datetime Conversion

In [55]:
datetime_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [56]:
for col in datetime_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [57]:
orders[datetime_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# Duplicate Removal

In [58]:
def remove_duplicates(df, name):
    before = len(df)

    df = df.drop_duplicates()

    removed = before - len(df)

    print(f"{name:<20} Removed {removed} duplicate rows")

    return df

In [59]:
customers = remove_duplicates(customers, "Customers")
orders = remove_duplicates(orders, "Orders")
order_items = remove_duplicates(order_items, "Order Items")
payments = remove_duplicates(payments, "Payments")
products = remove_duplicates(products, "Products")
reviews = remove_duplicates(reviews, "Reviews")
sellers = remove_duplicates(sellers, "Sellers")
geolocation = remove_duplicates(geolocation, "Geolocation")

Customers            Removed 0 duplicate rows
Orders               Removed 0 duplicate rows
Order Items          Removed 0 duplicate rows
Payments             Removed 0 duplicate rows
Products             Removed 0 duplicate rows
Reviews              Removed 0 duplicate rows
Sellers              Removed 0 duplicate rows
Geolocation          Removed 0 duplicate rows


# Negative Values

In [60]:
def replace_negative_with_nan(df, column):
    df.loc[df[column] < 0, column] = np.nan
    return df

In [61]:
payments = replace_negative_with_nan(
    payments,
    "payment_value"
)

order_items = replace_negative_with_nan(
    order_items,
    "price"
)

order_items = replace_negative_with_nan(
    order_items,
    "freight_value"
)

# Product Dimensions

In [62]:
def replace_non_positive_with_nan(df, column):
    df.loc[df[column] <= 0, column] = np.nan
    return df

In [63]:
dimension_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in dimension_columns:
    products = replace_non_positive_with_nan(
        products,
        column
    )

# Review Scores

In [64]:
reviews.loc[
    reviews["review_score"].between(1,5),
    "review_score"
] = np.nan

In [65]:
payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,323.610918
std,0.706584,2.687051,456.737502
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,119.260000
50%,1.000000,1.000000,210.000000
75%,1.000000,4.000000,360.855000
max,29.000000,24.000000,28694.570000


# Save Cleaned Data

In [66]:
datasets = {
    "customers.csv": customers,
    "orders.csv": orders,
    "order_items.csv": order_items,
    "payments.csv": payments,
    "products.csv": products,
    "reviews.csv": reviews,
    "sellers.csv": sellers,
    "geolocation.csv": geolocation,
}

for filename, df in datasets.items():
    df.to_csv(PROCESSED / filename, index=False)

print("✅ Cleaned datasets saved successfully.")

✅ Cleaned datasets saved successfully.
